# Analisis del audit de indices de vistas (`audit_view_indices.py`)

Lee `audit_view_indices_raw.csv` y `audit_view_indices_summary.csv` (generados en
el servidor con `Mapping/audit_view_indices.py`) y responde: **cuantos
experimentos usaron el patron de vistas SECUENCIAL (bug pre-fix de
`get_n_views_entries`, commit `a6ffeac` 2026-06-16) vs el patron CORRECTO
(espaciado por `linspace`)**, desglosado por prompt, experimento y n_views.

No hace falta abrir el CSV crudo a mano -- este notebook resume todo.

In [1]:
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

RAW_PATH     = "audit_view_indices_raw.csv"
SUMMARY_PATH = "audit_view_indices_summary.csv"

raw = pd.read_csv(RAW_PATH)
summary = pd.read_csv(SUMMARY_PATH)

print(f"raw:     {len(raw):,} filas (una por archivo x n_views)")
print(f"summary: {len(summary):,} filas (una por symmetry_type x prompt_id x experiment_id x n_views)")

raw:     95,200 filas (una por archivo x n_views)
summary: 112 filas (una por symmetry_type x prompt_id x experiment_id x n_views)


## 1. Veredicto global

In [2]:
counts = raw["clasificacion"].value_counts()
total = len(raw)

print("Clasificacion global (todas las combinaciones object_id x size x lighting x n_views x prompt/experimento):")
for label, n in counts.items():
    print(f"  {label:24s} {n:>8,}  ({n/total*100:5.1f}%)")

n_bug   = counts.get("secuencial (bug)", 0)
n_otro  = counts.get("otro/inesperado", 0)
n_ok    = counts.get("correcto (espaciado)", 0)
n_na    = counts.get("n/a (n_views>=total)", 0)

print()
if n_bug == 0 and n_otro == 0:
    print(f"VEREDICTO: ningun archivo del dataset auditado tiene el bug secuencial. "
          f"{n_ok:,}/{total:,} corridas ({n_ok/total*100:.1f}%) usan el patron correcto "
          f"(el resto, si hay, es n/a por n_views>=total_vistas).")
else:
    print(f"VEREDICTO: hay {n_bug:,} corridas con el bug secuencial y {n_otro:,} con un patron "
          f"inesperado -- ver secciones siguientes para identificar cuales.")

Clasificacion global (todas las combinaciones object_id x size x lighting x n_views x prompt/experimento):
  correcto (espaciado)       95,200  (100.0%)

VEREDICTO: ningun archivo del dataset auditado tiene el bug secuencial. 95,200/95,200 corridas (100.0%) usan el patron correcto (el resto, si hay, es n/a por n_views>=total_vistas).


## 2. Alcance del audit (para saber si cubrio todo lo esperado)

In [3]:
print("Tipos de simetria      :", sorted(raw["symmetry_type"].unique()))
print("Objetos unicos         :", raw["object_id"].nunique())
print("Objetos por tipo       :")
print(raw.groupby("symmetry_type")["object_id"].nunique().to_string())
print()
print("Grupos de n_views      :", sorted(raw["n_views"].unique()))
print("Sizes                  :", sorted(raw["size"].unique()))
print("Lightings              :", sorted(raw["lighting"].unique()))
print()
combos = raw[["prompt_id", "experiment_id"]].drop_duplicates().sort_values(["prompt_id", "experiment_id"])
print(f"Combinaciones (prompt_id, experiment_id) distintas: {len(combos)}")
print(combos.to_string(index=False))

Tipos de simetria      : ['axis_sym', 'plane_sym']
Objetos unicos         : 1700
Objetos por tipo       :
symmetry_type
axis_sym     850
plane_sym    850

Grupos de n_views      : [np.int64(1), np.int64(6), np.int64(14), np.int64(26)]
Sizes                  : [np.int64(224)]
Lightings              : ['flat']

Combinaciones (prompt_id, experiment_id) distintas: 28
  prompt_id     experiment_id
   axis_v00          axis_v00
 axis_v00_1        axis_v00_1
   axis_v01          axis_v01
 axis_v01_1        axis_v01_1
   axis_v02          axis_v02
 axis_v02_1        axis_v02_1
   axis_v03          axis_v03
 axis_v03_1        axis_v03_1
   axis_v04          axis_v04
 axis_v04_1        axis_v04_1
   axis_v05          axis_v05
 axis_v05_1        axis_v05_1
 axis_v05_1  axis_v05_1_flowB
 axis_v05_1  axis_v05_1_flowC
  plane_v00         plane_v00
plane_v00_1       plane_v00_1
  plane_v01         plane_v01
plane_v01_1       plane_v01_1
  plane_v02         plane_v02
plane_v02_1       plane_v02_1
  pl

## 3. Confiabilidad del chequeo: de donde salio el `total_vistas` usado

Si `total_fuente` es `fallback_default` en vez de `metadata_all.json`, la
clasificacion de esa fila se baso en un supuesto (114 vistas), no en el dato
real de esa carpeta -- conviene revisar esos casos a mano si aparecen.

In [4]:
fuente_counts = raw["total_fuente"].value_counts()
print(fuente_counts.to_string())

n_fallback = fuente_counts.get("fallback_default", 0)
if n_fallback:
    print(f"\n[atencion] {n_fallback:,} filas usaron el total por defecto -- carpetas sin metadata_all.json:")
    display_cols = ["symmetry_type", "object_id", "size", "lighting"]
    print(raw.loc[raw["total_fuente"] == "fallback_default", display_cols]
          .drop_duplicates()
          .to_string(index=False))
else:
    print("\nTodas las filas usaron el total real de metadata_all.json -- sin supuestos.")

total_fuente
metadata_all.json    95200

Todas las filas usaron el total real de metadata_all.json -- sin supuestos.


## 4. Desglose por prompt x experimento x n_views

Tabla completa (la misma info que `audit_view_indices_summary.csv`, ordenada
para que lo primero que se vea sean los casos problematicos si los hay).

In [5]:
summary_sorted = summary.sort_values("pct_correcto")
print(summary_sorted.to_string(index=False))

symmetry_type   prompt_id     experiment_id  n_views  secuencial (bug)  correcto (espaciado)  otro/inesperado  n_total_checkeado  pct_correcto
     axis_sym    axis_v00          axis_v00        1                 0                   850                0                850         100.0
     axis_sym    axis_v00          axis_v00        6                 0                   850                0                850         100.0
     axis_sym    axis_v00          axis_v00       14                 0                   850                0                850         100.0
     axis_sym    axis_v00          axis_v00       26                 0                   850                0                850         100.0
     axis_sym  axis_v00_1        axis_v00_1        1                 0                   850                0                850         100.0
     axis_sym  axis_v00_1        axis_v00_1        6                 0                   850                0                850         100.0

## 5. Filas problematicas (pct_correcto < 100%) -- si esta vacio, no hay nada que arreglar

In [6]:
problematic = summary[summary["pct_correcto"] < 100]

if problematic.empty:
    print("Ninguna combinacion (prompt_id, experiment_id, n_views) tiene corridas con el bug "
          "secuencial ni con un patron inesperado. Todo el dataset auditado usa vistas correctas.")
else:
    print(f"{len(problematic)} combinaciones con al menos una corrida problematica:\n")
    print(problematic.to_string(index=False))

    print("\nObjetos especificos afectados (para saber que regenerar):")
    bad_keys = problematic[["symmetry_type", "prompt_id", "experiment_id", "n_views"]]
    bad_files = raw.merge(bad_keys, on=["symmetry_type", "prompt_id", "experiment_id", "n_views"])
    bad_files = bad_files[bad_files["clasificacion"] != "correcto (espaciado)"]
    cols = ["symmetry_type", "object_id", "size", "lighting", "file",
            "prompt_id", "experiment_id", "n_views", "clasificacion"]
    print(bad_files[cols].to_string(index=False))
    bad_files[cols].to_csv("audit_view_indices_PROBLEMATICOS.csv", index=False)
    print("\nGuardado en audit_view_indices_PROBLEMATICOS.csv para regenerar solo esos.")

Ninguna combinacion (prompt_id, experiment_id, n_views) tiene corridas con el bug secuencial ni con un patron inesperado. Todo el dataset auditado usa vistas correctas.


## 6. Cobertura por n_views (por si algun prompt no llego a correr todos los grupos)

In [7]:
n_objects_by_type = raw.groupby("symmetry_type")["object_id"].nunique().to_dict()

coverage = (
    raw.groupby(["symmetry_type", "prompt_id", "experiment_id", "n_views"])["object_id"]
    .nunique()
    .reset_index(name="n_objetos_cubiertos")
)
coverage["n_objetos_esperados"] = coverage["symmetry_type"].map(n_objects_by_type)
coverage["pct_cobertura"] = (coverage["n_objetos_cubiertos"] / coverage["n_objetos_esperados"] * 100).round(1)

incompletos = coverage[coverage["pct_cobertura"] < 100].sort_values("pct_cobertura")
if incompletos.empty:
    print("Todas las combinaciones prompt x experimento x n_views cubren todos los objetos de su symmetry_type:")
    print(n_objects_by_type)
else:
    print("Combinaciones que NO cubren todos los objetos de su symmetry_type "
          "(puede ser una corrida parcial, no necesariamente un problema):\n")
    print(incompletos.to_string(index=False))


Todas las combinaciones prompt x experimento x n_views cubren todos los objetos de su symmetry_type:
{'axis_sym': 850, 'plane_sym': 850}


## Conclusion

Completar a mano despues de correr las celdas de arriba con los datos reales
del servidor -- las secciones 1 y 5 ya dan la respuesta directa a "cuantos
experimentos tienen el bug secuencial": si la seccion 5 sale vacia, la
respuesta es cero, y los resultados agregados de todos los experimentos son
confiables respecto a este problema especifico.